In [1]:
# ==========================================================
# PURCHASE PATTERN ANALYSIS - FINAL INTEGRATED VERSION
# ==========================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
from sqlalchemy import create_engine
from mlxtend.frequent_patterns import apriori, association_rules

# ==========================================================
# CREATE PROJECT OUTPUT FOLDERS (AUTO BACKUP SYSTEM)
# ==========================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base_folder = f"Purchase_Pattern_Output_{timestamp}"

charts_folder = os.path.join(base_folder, "charts")
data_folder = os.path.join(base_folder, "data_backup")

os.makedirs(charts_folder, exist_ok=True)
os.makedirs(data_folder, exist_ok=True)

print(f"Output Folder Created: {base_folder}")

# ==========================================================
# LOAD DATA FROM MYSQL
# ==========================================================

engine = create_engine(
    "mysql+pymysql://UserName:Password@ Host:Port/project_purchase_pattern_analysis"
)

print("\nConnecting to Database...")
df = pd.read_sql("SELECT * FROM mytable", engine)

# Standardize column names
df.columns = df.columns.str.strip().str.lower()

print("Data Loaded Successfully ✅")
print("Initial Dataset Shape:", df.shape)
print("Columns:", df.columns)

# ==========================================================
# BASIC EDA (KEY STATISTICS)
# ==========================================================

print("\n===== BASIC STATISTICS =====")

mean_qty = df['quantity'].mean()
median_qty = df['quantity'].median()
mode_qty = df['quantity'].mode()

print("Mean Quantity:", mean_qty)
print("Median Quantity:", median_qty)
print("Mode Quantity:", mode_qty.iloc[0])
print("\nFull Statistical Summary:")
print(df.describe())

# ==========================================================
# DATA CLEANING
# ==========================================================

print("\n===== DATA CLEANING =====")

df = df.drop_duplicates()
df = df[~df['billno'].astype(str).str.startswith('C')]
df = df[df['quantity'] > 0]
df = df.dropna(subset=['itemname'])

df['itemname'] = df['itemname'].astype(str).str.strip().str.upper()

print("Data Cleaning Completed ✅")
print("Cleaned Dataset Shape:", df.shape)

# Save cleaned dataset
df.to_csv(os.path.join(data_folder, "cleaned_transactions.csv"), index=False)

# ==========================================================
# VISUALIZATION SECTION
# ==========================================================

sns.set(style="whitegrid")

# ----------------------------------------------------------
# Top 10 Most Sold Products
# ----------------------------------------------------------

top_items = df['itemname'].value_counts().head(10)

plt.figure(figsize=(10,6))
sns.barplot(
    x=top_items.values,
    y=top_items.index,
    palette="viridis"
)

plt.title("Top 10 Most Sold Products")
plt.xlabel("Count")
plt.ylabel("Product")
plt.tight_layout()
plt.savefig(os.path.join(charts_folder, "top_10_products.png"), dpi=300)
plt.close()

# ----------------------------------------------------------
# Items Per Transaction Distribution
# ----------------------------------------------------------

items_per_bill = df.groupby('billno')['itemname'].count()

plt.figure(figsize=(8,5))
sns.histplot(items_per_bill, bins=20, kde=True, color="orange")

plt.title("Items per Transaction Distribution")
plt.xlabel("Number of Items")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(charts_folder, "items_per_transaction.png"), dpi=300)
plt.close()

# ==========================================================
# MARKET BASKET ANALYSIS (APRIORI)
# ==========================================================

print("\n===== APPLYING APRIORI ALGORITHM =====")

basket = (
    df.groupby(['billno', 'itemname'])['quantity']
    .sum()
    .unstack()
    .fillna(0)
)

basket = basket > 0

print("Basket Shape:", basket.shape)

# Frequent Itemsets
frequent_itemsets = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

print("Frequent Itemsets Generated ✅")
print("Number of Frequent Itemsets:", len(frequent_itemsets))

frequent_itemsets.to_csv(
    os.path.join(data_folder, "frequent_itemsets.csv"),
    index=False
)

# ----------------------------------------------------------
# Association Rules
# ----------------------------------------------------------

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3
)

print("Association Rules Generated ✅")
print("Total Rules:", len(rules))

rules.to_csv(
    os.path.join(data_folder, "association_rules.csv"),
    index=False
)

# ==========================================================
# TOP 10 RULES BY LIFT
# ==========================================================

top_rules = rules.sort_values(by='lift', ascending=False).head(10)

# Create readable rule text
top_rules['rule'] = (
    top_rules['antecedents'].astype(str) + " → " +
    top_rules['consequents'].astype(str)
)

plt.figure(figsize=(10,6))
sns.barplot(
    x=top_rules['lift'],
    y=top_rules['rule'],
    palette="coolwarm"
)

plt.title("Top 10 Rules by Lift")
plt.xlabel("Lift")
plt.ylabel("Rule")
plt.tight_layout()
plt.savefig(os.path.join(charts_folder, "top_10_rules_by_lift.png"), dpi=300)
plt.close()

# ==========================================================
# SUPPORT VS CONFIDENCE SCATTER
# ==========================================================

plt.figure(figsize=(8,6))

scatter = plt.scatter(
    rules['support'],
    rules['confidence'],
    s=rules['lift'] * 100,
    c=rules['lift'],
    cmap="Spectral",
    alpha=0.7
)

plt.colorbar(scatter, label="Lift")
plt.title("Support vs Confidence")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.tight_layout()
plt.savefig(os.path.join(charts_folder, "support_vs_confidence.png"), dpi=300)
plt.close()

# ==========================================================
# FINAL SUMMARY OUTPUT
# ==========================================================

print("\n===== PROJECT SUMMARY =====")
print("Total Transactions:", df['billno'].nunique())
print("Total Unique Products:", df['itemname'].nunique())
print("Top Product:", top_items.index[0])
print("Highest Lift Value:", round(top_rules['lift'].max(), 2))

print("\nPROJECT COMPLETED SUCCESSFULLY 🚀")
print(f"All outputs saved inside folder: {base_folder}")

D:\Users\shali\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Output Folder Created: Purchase_Pattern_Output_20260228_002753

Connecting to Database...
Data Loaded Successfully ✅
Initial Dataset Shape: (522064, 7)
Columns: Index(['billno', 'itemname', 'quantity', 'present_date', 'price', 'customerid',
       'country'],
      dtype='str')

===== BASIC STATISTICS =====
Mean Quantity: 10.090435272303779
Median Quantity: 3.0
Mode Quantity: 1

Full Statistical Summary:
            quantity          price
count  522064.000000  522064.000000
mean       10.090435       3.826801
std       161.110525      41.900599
min     -9600.000000  -11062.060000
25%         1.000000       1.250000
50%         3.000000       2.080000
75%        10.000000       4.130000
max     80995.000000   13541.330000

===== DATA CLEANING =====
Data Cleaning Completed ✅
Cleaned Dataset Shape: (515442, 7)


C:\Users\shali\AppData\Local\Temp\ipykernel_31136\2831326081.py:94: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(



===== APPLYING APRIORI ALGORITHM =====
Basket Shape: (20327, 4052)
Frequent Itemsets Generated ✅
Number of Frequent Itemsets: 357
Association Rules Generated ✅
Total Rules: 140


C:\Users\shali\AppData\Local\Temp\ipykernel_31136\2831326081.py:186: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
C:\Users\shali\AppData\Local\Temp\ipykernel_31136\2831326081.py:195: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()



===== PROJECT SUMMARY =====
Total Transactions: 20327
Total Unique Products: 4052
Top Product: WHITE HANGING HEART T-LIGHT HOLDER
Highest Lift Value: 19.46

PROJECT COMPLETED SUCCESSFULLY 🚀
All outputs saved inside folder: Purchase_Pattern_Output_20260228_002753
